In [ ]:
from pyspark.sql.functions import col

from olist_silver.transformations import (
    is_valid_uuid,
    merge_into,
    with_processed_timestamp,
)

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_sellers_table_name = dbutils.widgets.get("raw_olist_sellers_table")

silver_schema = dbutils.widgets.get("silver_schema")
sellers_table_name = dbutils.widgets.get("sellers_table")

In [ ]:
raw_olist_sellers_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_sellers_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{sellers_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{sellers_table_name} (
            sellerId STRING,
            sellerZipCodePrefix STRING,
            sellerCity STRING,
            sellerState STRING,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
sellers_silver_df = with_processed_timestamp(
    raw_olist_sellers_df.where(is_valid_uuid("seller_id"))
    .select(
        col("seller_id").cast("string").alias("sellerId"),
        col("seller_zip_code_prefix").cast("string").alias("sellerZipCodePrefix"),
        col("seller_city").cast("string").alias("sellerCity"),
        col("seller_state").cast("string").alias("sellerState"),
    )
    .dropDuplicates(["sellerId"])
)

In [ ]:
merge_into(
    spark,
    target=f"{catalog}.{silver_schema}.{sellers_table_name}",
    source_view="sellers_silver_view",
    keys=["sellerId"],
    source_df=sellers_silver_df,
)